# 02 Feature Engineering

This notebook validates interpretable feature engineering for the Customer Personality Analysis project.

Scope for this step:
- Create demographic, spending, purchase, and historical campaign behavior features.
- Preserve the readable customer-level dataset for later EDA, clustering, and supervised modeling.
- Save `data/processed/customer_features.csv`.

This notebook does not scale variables, one-hot encode categories, create final target variables, perform clustering, or train models.

## Setup

The reusable feature engineering logic lives in `src/feature_engineering.py`. This notebook imports those functions so the validation matches the production script.

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.append(str(SRC_PATH))

from feature_engineering import (
    CAMPAIGN_COLUMNS,
    CLEAN_DATA_PATH,
    ENGINEERED_FEATURES,
    FEATURE_DATA_PATH,
    PURCHASE_CHANNEL_COLUMNS,
    SPENDING_COLUMNS,
    create_customer_features,
    load_clean_data,
    save_feature_data,
    validate_features,
)

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 140)

print(f"Project root: {PROJECT_ROOT}")
print(f"Clean data path: {CLEAN_DATA_PATH}")
print(f"Feature data path: {FEATURE_DATA_PATH}")

Project root: /Users/arjun/Documents/Intro DS final
Clean data path: /Users/arjun/Documents/Intro DS final/data/processed/customer_clean.csv
Feature data path: /Users/arjun/Documents/Intro DS final/data/processed/customer_features.csv


## Inspect the Cleaned Dataset

Before creating features, confirm the input shape, columns, date range, and missing value status.

In [2]:
clean_df = load_clean_data(CLEAN_DATA_PATH)

print(f"Input dataframe shape: {clean_df.shape}")
print(f"Date range: {clean_df['dt_customer'].min().date()} to {clean_df['dt_customer'].max().date()}")
print(f"Missing values in input: {int(clean_df.isna().sum().sum())}")

display(clean_df.head())

Input dataframe shape: (2213, 29)
Date range: 2012-07-30 to 2014-06-29
Missing values in input: 0


,id,year_birth,education,marital_status,income,kidhome,teenhome,dt_customer,recency,mnt_wines,mnt_fruits,mnt_meat_products,mnt_fish_products,mnt_sweet_products,mnt_gold_prods,num_deals_purchases,num_web_purchases,num_catalog_purchases,num_store_purchases,num_web_visits_month,accepted_cmp3,accepted_cmp4,accepted_cmp5,accepted_cmp1,accepted_cmp2,complain,z_cost_contact,z_revenue,response
0,5524,1957,Graduation,Single,58138.0,0,0,2012-09-04,58,635,88,546,172,88,88,3,8,10,4,7,0,0,0,0,0,0,3,11,1
1,2174,1954,Graduation,Single,46344.0,1,1,2014-03-08,38,11,1,6,2,1,6,2,1,1,2,5,0,0,0,0,0,0,3,11,0
2,4141,1965,Graduation,Together,71613.0,0,0,2013-08-21,26,426,49,127,111,21,42,1,8,2,10,4,0,0,0,0,0,0,3,11,0
3,6182,1984,Graduation,Together,26646.0,1,0,2014-02-10,26,11,4,20,10,3,5,2,2,0,4,6,0,0,0,0,0,0,3,11,0
4,5324,1981,PhD,Married,58293.0,1,0,2014-01-19,94,173,43,118,46,27,15,5,5,3,6,5,0,0,0,0,0,0,3,11,0


In [3]:
input_summary = pd.DataFrame({
    "column": clean_df.columns,
    "dtype": [str(dtype) for dtype in clean_df.dtypes],
    "missing_values": clean_df.isna().sum().to_numpy(),
})
display(input_summary)

print("Marital status values used for household-size assumptions")
display(clean_df["marital_status"].value_counts().to_frame("rows"))

,column,dtype,missing_values
0,id,int64,0
1,year_birth,int64,0
2,education,str,0
3,marital_status,str,0
4,income,float64,0
5,kidhome,int64,0
6,teenhome,int64,0
7,dt_customer,datetime64[us],0
8,recency,int64,0
9,mnt_wines,int64,0


Marital status values used for household-size assumptions


,rows
marital_status,
Married,857
Together,572
Single,470
Divorced,231
Widow,76
Alone,3
Absurd,2
YOLO,2


## Feature Creation

The feature script uses the latest `dt_customer` value as the dataset reference date. This keeps age and tenure tied to the historical dataset instead of changing when the notebook is rerun in a future calendar year.

In [4]:
feature_df, reference_date = create_customer_features(clean_df)

print(f"Reference date: {reference_date.date()}")
print(f"Before shape: {clean_df.shape}")
print(f"After shape: {feature_df.shape}")
print(f"New columns added: {feature_df.shape[1] - clean_df.shape[1]}")

Reference date: 2014-06-29
Before shape: (2213, 29)
After shape: (2213, 53)
New columns added: 24


## Demographic Features

Demographic features include age, customer tenure, total children at home, a binary child flag, and a conservative household-size estimate.

Household-size assumption:
- `Married` and `Together` are treated as two-adult households.
- All other marital statuses are treated as one-adult households because the data does not prove another adult is present.

In [5]:
demographic_features = [
    "age",
    "customer_tenure_days",
    "customer_tenure_years",
    "children_total",
    "has_children",
    "household_size",
]

display(feature_df[demographic_features].describe().T)
display(feature_df[["id", "year_birth", "dt_customer", "marital_status", "kidhome", "teenhome"] + demographic_features].head())

,count,mean,std,min,25%,50%,75%,max
age,2213.0,45.082693,11.700216,18.0,37.000000,44.000000,55.000000,74.000000
customer_tenure_days,2213.0,353.731586,202.450745,0.0,180.000000,356.000000,529.000000,699.000000
customer_tenure_years,2213.0,0.968464,0.554280,0.0,0.492813,0.974675,1.448323,1.913758
children_total,2213.0,0.947582,0.749297,0.0,0.000000,1.000000,1.000000,3.000000
has_children,2213.0,0.714415,0.451795,0.0,0.000000,1.000000,1.000000,1.000000
household_size,2213.0,2.593312,0.906073,1.0,2.000000,3.000000,3.000000,5.000000


,id,year_birth,dt_customer,marital_status,kidhome,teenhome,age,customer_tenure_days,customer_tenure_years,children_total,has_children,household_size
0,5524,1957,2012-09-04,Single,0,0,57,663,1.815195,0,0,1
1,2174,1954,2014-03-08,Single,1,1,60,113,0.309377,2,1,3
2,4141,1965,2013-08-21,Together,0,0,49,312,0.854209,0,0,2
3,6182,1984,2014-02-10,Together,1,0,30,139,0.380561,1,1,3
4,5324,1981,2014-01-19,Married,1,0,33,161,0.440794,1,1,3


## Spending Behavior Features

Spending features summarize total product spending, product-category shares, a wine/gold luxury-oriented ratio, and annualized spending based on customer tenure. Ratio calculations return 0 when the denominator is 0 to avoid missing or infinite values.

In [6]:
spending_features = [
    "total_spending",
    "wine_share",
    "fruits_share",
    "meat_share",
    "fish_share",
    "sweets_share",
    "gold_share",
    "luxury_spending_ratio",
    "spending_per_tenure_year",
]

display(feature_df[spending_features].describe().T)

share_check = feature_df[["wine_share", "fruits_share", "meat_share", "fish_share", "sweets_share", "gold_share"]].sum(axis=1)
print("Spending share sum range:", share_check.min(), "to", share_check.max())
display(feature_df[["id"] + SPENDING_COLUMNS + spending_features].head())

,count,mean,std,min,25%,50%,75%,max
total_spending,2213.0,607.021690,602.488663,5.000000,69.000000,397.000000,1048.000000,2525.000000
wine_share,2213.0,0.458868,0.228649,0.000000,0.289803,0.458115,0.641164,0.963303
fruits_share,2213.0,0.049542,0.055889,0.000000,0.008989,0.029830,0.070175,0.445545
meat_share,2213.0,0.249118,0.125752,0.000000,0.156250,0.233333,0.328221,0.997110
fish_share,2213.0,0.071638,0.078028,0.000000,0.012575,0.048193,0.104925,0.590909
sweets_share,2213.0,0.050774,0.060945,0.000000,0.008631,0.033333,0.070707,0.945848
gold_share,2213.0,0.120060,0.108818,0.000000,0.038038,0.085714,0.169811,0.894150
luxury_spending_ratio,2213.0,0.578929,0.194040,0.001156,0.428714,0.571429,0.737844,0.972477
spending_per_tenure_year,2213.0,2222.856039,15492.132710,4.131787,113.958000,453.979617,1107.532258,359406.000000


Spending share sum range: 0.9999999999999998 to 1.0000000000000002


,id,mnt_wines,mnt_fruits,mnt_meat_products,mnt_fish_products,mnt_sweet_products,mnt_gold_prods,total_spending,wine_share,fruits_share,meat_share,fish_share,sweets_share,gold_share,luxury_spending_ratio,spending_per_tenure_year
0,5524,635,88,546,172,88,88,1617,0.392703,0.054422,0.337662,0.106370,0.054422,0.054422,0.447124,890.813348
1,2174,11,1,6,2,1,6,27,0.407407,0.037037,0.222222,0.074074,0.037037,0.222222,0.629630,87.272124
2,4141,426,49,127,111,21,42,776,0.548969,0.063144,0.163660,0.143041,0.027062,0.054124,0.603093,908.442308
3,6182,11,4,20,10,3,5,53,0.207547,0.075472,0.377358,0.188679,0.056604,0.094340,0.301887,139.267986
4,5324,173,43,118,46,27,15,422,0.409953,0.101896,0.279621,0.109005,0.063981,0.035545,0.445498,957.363354


## Purchase Behavior Features

Purchase features summarize web, catalog, and store purchase activity separately from deal purchases. `average_spend_per_purchase` uses web/catalog/store purchases as the denominator because deal purchases may overlap with those channels.

In [7]:
purchase_features = [
    "total_purchases",
    "total_store_web_catalog_purchases",
    "web_purchase_share",
    "store_purchase_share",
    "catalog_purchase_share",
    "deal_purchase_share",
    "average_spend_per_purchase",
]

display(feature_df[purchase_features].describe().T)

channel_share_check = feature_df[["web_purchase_share", "store_purchase_share", "catalog_purchase_share"]].sum(axis=1)
print("Channel purchase share sum range:", channel_share_check.min(), "to", channel_share_check.max())
display(feature_df[["id"] + PURCHASE_CHANNEL_COLUMNS + ["num_deals_purchases"] + purchase_features].head())

,count,mean,std,min,25%,50%,75%,max
total_purchases,2213.0,14.889742,7.670341,0.0,8.000000,15.000000,21.000000,44.000000
total_store_web_catalog_purchases,2213.0,12.564392,7.204770,0.0,6.000000,12.000000,18.000000,32.000000
web_purchase_share,2213.0,0.328887,0.122083,0.0,0.250000,0.333333,0.400000,1.000000
store_purchase_share,2213.0,0.503673,0.151081,0.0,0.400000,0.500000,0.600000,1.000000
catalog_purchase_share,2213.0,0.164729,0.140594,0.0,0.000000,0.150000,0.250000,1.000000
deal_purchase_share,2213.0,0.180277,0.111247,0.0,0.076923,0.166667,0.250000,1.000000
average_spend_per_purchase,2213.0,37.436296,30.087924,0.0,13.000000,29.833333,49.166667,187.666667


Channel purchase share sum range: 0.0 to 1.0


,id,num_web_purchases,num_catalog_purchases,num_store_purchases,num_deals_purchases,total_purchases,total_store_web_catalog_purchases,web_purchase_share,store_purchase_share,catalog_purchase_share,deal_purchase_share,average_spend_per_purchase
0,5524,8,10,4,3,25,22,0.363636,0.181818,0.454545,0.120000,73.500000
1,2174,1,1,2,2,6,4,0.250000,0.500000,0.250000,0.333333,6.750000
2,4141,8,2,10,1,21,20,0.400000,0.500000,0.100000,0.047619,38.800000
3,6182,2,0,4,2,8,6,0.333333,0.666667,0.000000,0.250000,8.833333
4,5324,5,3,6,5,19,14,0.357143,0.428571,0.214286,0.263158,30.142857


## Campaign Behavior Features

These features summarize historical campaign acceptances from `accepted_cmp1` through `accepted_cmp5`. The recent campaign `response` column is intentionally left unchanged and is not used to create these features.

In [8]:
campaign_features = [
    "total_campaign_acceptances",
    "any_campaign_acceptance",
]

display(feature_df[campaign_features].describe().T)
display(feature_df[["id"] + CAMPAIGN_COLUMNS + campaign_features + ["response"]].head())

,count,mean,std,min,25%,50%,75%,max
total_campaign_acceptances,2213.0,0.298238,0.679446,0.0,0.0,0.0,0.0,4.0
any_campaign_acceptance,2213.0,0.206959,0.405217,0.0,0.0,0.0,0.0,1.0


,id,accepted_cmp1,accepted_cmp2,accepted_cmp3,accepted_cmp4,accepted_cmp5,total_campaign_acceptances,any_campaign_acceptance,response
0,5524,0,0,0,0,0,0,0,1
1,2174,0,0,0,0,0,0,0,0
2,4141,0,0,0,0,0,0,0,0
3,6182,0,0,0,0,0,0,0,0
4,5324,0,0,0,0,0,0,0,0


## Validation Checks

The engineered dataset should remain readable and should not contain missing or infinite values. Categorical variables are preserved for later EDA and modeling preparation; no one-hot encoding is performed in this file.

In [9]:
validation = validate_features(feature_df)
display(pd.Series(validation, name="count").to_frame())

remaining_missing = feature_df.isna().sum()
display(remaining_missing[remaining_missing > 0].to_frame("missing_values"))

numeric_df = feature_df.select_dtypes(include=[np.number])
infinite_count_by_column = pd.Series(
    np.isinf(numeric_df.to_numpy()).sum(axis=0),
    index=numeric_df.columns,
)
display(infinite_count_by_column[infinite_count_by_column > 0].to_frame("infinite_values"))

,count
rows,2213
columns,53
missing_values,0
infinite_values,0


,missing_values


,infinite_values


In [10]:
print("Engineered features created:")
for feature in ENGINEERED_FEATURES:
    print(f"- {feature}")

sample_columns = [
    "id",
    "age",
    "customer_tenure_years",
    "children_total",
    "household_size",
    "total_spending",
    "wine_share",
    "meat_share",
    "total_purchases",
    "average_spend_per_purchase",
    "total_campaign_acceptances",
    "response",
]
display(feature_df[sample_columns].head())

Engineered features created:
- age
- customer_tenure_days
- customer_tenure_years
- children_total
- has_children
- household_size
- total_spending
- wine_share
- fruits_share
- meat_share
- fish_share
- sweets_share
- gold_share
- luxury_spending_ratio
- spending_per_tenure_year
- total_purchases
- total_store_web_catalog_purchases
- web_purchase_share
- store_purchase_share
- catalog_purchase_share
- deal_purchase_share
- average_spend_per_purchase
- total_campaign_acceptances
- any_campaign_acceptance


,id,age,customer_tenure_years,children_total,household_size,total_spending,wine_share,meat_share,total_purchases,average_spend_per_purchase,total_campaign_acceptances,response
0,5524,57,1.815195,0,1,1617,0.392703,0.337662,25,73.500000,0,1
1,2174,60,0.309377,2,3,27,0.407407,0.222222,6,6.750000,0,0
2,4141,49,0.854209,0,2,776,0.548969,0.163660,21,38.800000,0,0
3,6182,30,0.380561,1,3,53,0.207547,0.377358,8,8.833333,0,0
4,5324,33,0.440794,1,3,422,0.409953,0.279621,19,30.142857,0,0


## Save Engineered Dataset

The final feature file is saved for later EDA, clustering, and supervised modeling steps.

In [11]:
save_feature_data(feature_df, FEATURE_DATA_PATH)
print(f"Saved engineered dataset to: {FEATURE_DATA_PATH}")

assert feature_df.isna().sum().sum() == 0
assert np.isinf(feature_df.select_dtypes(include=[np.number]).to_numpy()).sum() == 0
assert set(ENGINEERED_FEATURES).issubset(feature_df.columns)

Saved engineered dataset to: /Users/arjun/Documents/Intro DS final/data/processed/customer_features.csv
